# Sun et al. binary-age differential-expression comparison

This notebook:

1. Loads the downloaded Sun et al. coronal MERFISH AnnData object.
2. Restricts the data to the young and old age groups used in the manuscript.
3. Exports count matrices and metadata for ALDEx.
4. Postprocesses Sun et al. ALDEx outputs and loads primary MERFISH ALDEx results for comparison.
5. Compares effect sizes using sign concordance, correlations, and CAT@K.
6. Applies the adapted rank-based pseudobulk method to both datasets.
7. Repeats the cross-dataset comparison for the rank-based results.
8. Plots ALDEx and rank-based CAT@K jointly and calculates ΔCAT@K.
9. Writes and renders the supplemental effect-size concordance table.

Code has been streamlined since initial analysis but the scientific logic, age groups, anatomical mappings, normalization, tests, and common-gene comparison universe follow the original analysis.

## 1. Imports and configuration

In [ ]:
from __future__ import annotations

from glob import glob
from pathlib import Path
from typing import Iterable

import anndata as ad
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.stats import mannwhitneyu

import kimlabspatial.differential_expression as de

from scale_aware_st import (
    RepositoryConfig,
    index_result_tables,
    load_pooled_aldex_spec,
    load_pooled_strata,
)


# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------
config = RepositoryConfig.from_env()
ALDEX_RESULT_MODE = "published"  # choose "published" or "recompute"
PROJECT_DIR = config.root
DATA_DIR = config.data_dir
DE_INPUT_DIR = config.results_dir / "DEG" / "sun_binary_inputs"
DE_RESULT_DIR = config.results_dir / "DEG"
FIGURE_DIR = config.results_dir / "figures"
TABLE_DIR = config.results_dir / "supplementary_tables"

SUN_H5AD = config.external_data_dir / "sun_et_al" / "aging_coronal.h5ad"
PRIMARY_H5AD = config.data_dir / "primary_merfish" / "analysis_objects" / "adata_glia_aldex_pp.h5ad"

SUN_ALDEX_RAW_DIR = DE_RESULT_DIR / "cross_study_it_results"
SUN_ALDEX_RAW_GLOB = str(SUN_ALDEX_RAW_DIR / "cross_study_ct_*.xlsx")
SUN_ALDEX_RAW_PREFIX = str(SUN_ALDEX_RAW_DIR / "cross_study_ct_")
SUN_ALDEX_RAW_SUFFIX = "_it_results_02162026"
SUN_ALDEX_SIGNIFICANT = DE_RESULT_DIR / "cross_study1_inftss_results.xlsx"
SUN_ALDEX_ALL = DE_RESULT_DIR / "cross_study1_ALL_inftss_results.xlsx"
PRIMARY_ALDEX_ALL = DE_RESULT_DIR / "aldex_ALL_ct_anterior_inftss_results.xlsx"

for directory in (DE_INPUT_DIR, FIGURE_DIR, TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Analysis constants retained from the original notebook
# ---------------------------------------------------------------------
SUN_YOUNG_AGES = [5.4, 4.3, 3.8, 3.4]
SUN_OLD_AGES = [24.6, 23.5, 21.4, 19.8]
SUN_SELECTED_AGES = SUN_YOUNG_AGES + SUN_OLD_AGES

CAT_K_VALUES = list(range(5, 21))
DELTA_K_VALUES = [5, 9, 11, 16, 19, 20]

SUN_VALID_CELL_TYPES = [
    "Astrocyte",
    "Endothelial",
    "Microglia",
    "Neuron-Excitatory",
    "Neuron-Inhibitory",
    "Neuron-MSN",
    "OPC",
    "Oligodendrocyte",
    "Pericyte",
    "VSMC",
]

PRIMARY_TO_SUN_CELL_TYPE = {
    "Astro": "Astrocyte",
    "EC": "Endothelial",
    "Immune": "Microglia",
    "OLG": "Oligodendrocyte",
    "OPC": "OPC",
    "PC": "Pericyte",
    "VSMC": "VSMC",
}

PRIMARY_ALDEX_TO_SUN_CELL_TYPE = {
    "astro": "astrocyte",
    "ec": "endothelial",
    "immune": "microglia",
    "olg": "oligodendrocyte",
    "opc": "opc",
    "pc": "pericyte",
    "vsmc": "vsmc",
}

PRIMARY_TO_SUN_REGION = {
    "all": "all",
    "Isocortex": "CTX",
    "cerebralnuclei": "STR",
}

## 2. Helper functions

In [ ]:

def postprocess_aldex2(
    aldex_results: Iterable[str],
    output_dir: Path,
    filename_prefix: str,
    filename_suffix: str,
    output_stem: str,
    experimental_column: str,
    pivot_filename: str | None,
    *,
    significant: bool = True,
    make_pivot: bool = True,
    p_threshold: float = 0.05,
    signed: bool = False,
) -> dict[str, pd.DataFrame]:
    """
    Postprocess Sun et al. per-cell-type ALDEx workbooks.

    This retains the original filename parsing, adjusted-p-value filtering,
    multi-sheet workbook output, and optional signed pivot summary.
    """
    aldex_results = list(aldex_results)
    output_dir.mkdir(parents=True, exist_ok=True)

    result_tables: dict[str, pd.DataFrame] = {}

    for result_path in aldex_results:
        result_string = str(result_path)
        start = result_string.find(filename_prefix) + len(filename_prefix)
        end = result_string.find(filename_suffix)

        if start < len(filename_prefix) or end < 0:
            raise ValueError(
                f"Could not parse cell type/region label from: {result_path}"
            )

        label = result_string[start:end]
        table = pd.read_excel(result_path, engine="openpyxl")

        if significant:
            table = table[
                table[f"{experimental_column}:pval.adj"] <= p_threshold
            ].copy()
        else:
            table = table.copy()

        result_tables[label] = table

    write_excel_sheets(
        result_tables,
        output_dir / f"{output_stem}.xlsx",
    )

    if make_pivot:
        combined_genes = pd.concat(
            [table[["gene"]] for table in result_tables.values()],
            ignore_index=True,
        )
        gene_counts = (
            combined_genes.value_counts()
            .reset_index(name="count")
            .set_index("gene")
        )
        gene_counts.index.name = None

        for label, table in result_tables.items():
            if signed:
                effect_map = table.set_index("gene")[f"{experimental_column}:est"]
                gene_counts[label] = gene_counts.index.map(
                    lambda gene: (
                        "+"
                        if gene in effect_map.index and effect_map.loc[gene] > 0
                        else "-"
                        if gene in effect_map.index and effect_map.loc[gene] < 0
                        else 0
                    )
                )
            else:
                gene_counts[label] = gene_counts.index.isin(
                    table["gene"]
                ).astype(int)

        if signed:
            result_columns = [
                column for column in gene_counts.columns
                if column != "count"
            ]
            positive = gene_counts[result_columns].eq("+").sum(axis=1)
            negative = gene_counts[result_columns].eq("-").sum(axis=1)
            insert_at = gene_counts.columns.get_loc("count") + 1
            gene_counts.insert(insert_at, "positive", positive)
            gene_counts.insert(insert_at + 1, "negative", negative)
            totals = positive + negative
            gene_counts.insert(
                insert_at + 2,
                "sign_agreement",
                (
                    gene_counts[["positive", "negative"]].max(axis=1)
                    / totals
                    * 100
                ).fillna(0),
            )

        if pivot_filename is None:
            raise ValueError(
                "pivot_filename is required when make_pivot=True."
            )
        gene_counts.to_excel(output_dir / pivot_filename)

    return result_tables


def read_excel_sheets(
    workbook: Path,
    *,
    index_column: str = "gene",
) -> dict[str, pd.DataFrame]:
    """Read all workbook sheets and index each table by gene."""
    results = {}
    for sheet_name in pd.ExcelFile(workbook).sheet_names:
        table = pd.read_excel(workbook, sheet_name=sheet_name)
        unnamed = [
            column for column in table.columns
            if str(column).startswith("Unnamed:")
        ]
        if unnamed:
            table = table.drop(columns=unnamed)
        table = table.set_index(index_column)
        table.index.name = None
        results[sheet_name] = table
    return results


def write_excel_sheets(
    tables: dict[str, pd.DataFrame],
    output_path: Path,
) -> None:
    """Write one DataFrame per Excel worksheet."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(output_path) as writer:
        for sheet_name, table in tables.items():
            table.to_excel(writer, sheet_name=sheet_name[:31])


def build_aldex_comparisons(
    primary_results: dict[str, pd.DataFrame],
    sun_results: dict[str, pd.DataFrame],
) -> tuple[
    dict[str, pd.DataFrame | None],
    dict[str, list[str] | None],
]:
    """
    Match primary cell type × region results to Sun et al. results.

    The region and cell-type mappings are retained from the original notebook.
    """
    comparisons = {key: None for key in primary_results}
    common_genes = {key: None for key in primary_results}

    for key, primary_df in primary_results.items():
        parts = key.split("_")
        if len(parts) < 2:
            continue

        primary_region = parts[0]
        primary_cell_type = parts[1]

        if primary_region not in PRIMARY_TO_SUN_REGION:
            continue
        if primary_cell_type not in PRIMARY_ALDEX_TO_SUN_CELL_TYPE:
            continue

        sun_region = PRIMARY_TO_SUN_REGION[primary_region]
        sun_cell_type = PRIMARY_ALDEX_TO_SUN_CELL_TYPE[primary_cell_type]
        sun_key = f"{sun_region}_{sun_cell_type}"

        if sun_key not in sun_results:
            continue

        sun_df = sun_results[sun_key]
        shared = sorted(set(primary_df.index) & set(sun_df.index))
        common_genes[key] = shared

        comparison = pd.concat(
            [
                primary_df.loc[shared, "age_binary:est"],
                sun_df.loc[shared, "age_binary:est"],
            ],
            axis=1,
        )
        comparison.columns = ["effect_ours", "effect_theirs"]
        comparison = comparison.dropna()

        if comparison.shape[0] >= 2:
            comparisons[key] = comparison

    return comparisons, common_genes


def make_sign_matrix(
    comparisons: dict[str, pd.DataFrame | None],
    *,
    key_filter: str | None = None,
) -> pd.DataFrame:
    """Create concordant/discordant/absent sign matrix."""
    valid = {
        key: table
        for key, table in comparisons.items()
        if table is not None and (key_filter is None or key_filter in key)
    }
    genes = sorted(set().union(*(table.index for table in valid.values())))
    matrix = pd.DataFrame(index=genes, columns=list(valid))

    for label, table in valid.items():
        product = np.sign(table["effect_ours"]) * np.sign(table["effect_theirs"])
        agreement = np.where(
            product > 0,
            1,
            np.where(product < 0, -1, np.nan),
        )
        matrix.loc[table.index, label] = agreement

    return matrix.T


def plot_sign_concordance(
    comparisons: dict[str, pd.DataFrame | None],
    *,
    title: str,
    output_path: Path,
    key_filter: str | None = None,
) -> pd.DataFrame:
    """Plot sign-concordance heatmap."""
    matrix = make_sign_matrix(comparisons, key_filter=key_filter)

    colors = ["red", "lightgrey", "green"]
    bounds = [-1.5, -0.5, 0.5, 1.5]
    cmap = ListedColormap(colors)
    cmap.set_bad(color="lightgrey")
    norm = BoundaryNorm(bounds, cmap.N)

    plt.figure(figsize=(15, 5))
    axis = sns.heatmap(
        matrix.astype(float),
        cmap=cmap,
        norm=norm,
        cbar=False,
        linewidths=0.1,
        linecolor="black",
    )
    axis.set_ylabel("Cell types", fontsize=14)
    axis.set_xlabel("Genes", fontsize=14)
    axis.xaxis.tick_top()
    axis.xaxis.set_label_position("top")
    plt.setp(axis.get_xticklabels(), rotation=90, ha="center", va="bottom")

    axis.legend(
        handles=[
            Patch(color="green", label="Concordant"),
            Patch(color="red", label="Discordant"),
            Patch(color="lightgrey", label="Absent"),
        ],
        loc="upper left",
        bbox_to_anchor=(1.05, 1),
        frameon=False,
        fontsize=14,
    )
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_path, dpi=600, bbox_inches="tight")
    plt.show()
    return matrix


def correlation_table(
    comparisons: dict[str, pd.DataFrame | None],
) -> pd.DataFrame:
    """Calculate Pearson and Spearman correlations."""
    records = []

    for label, table in comparisons.items():
        if table is None:
            continue

        table = table.dropna(subset=["effect_ours", "effect_theirs"])
        if len(table) <= 2:
            continue

        records.append(
            {
                "Cell type": label.replace("_", " ").title(),
                "N genes": int(len(table)),
                "Pearson r": table["effect_ours"].corr(
                    table["effect_theirs"],
                    method="pearson",
                ),
                "Spearman ρ": table["effect_ours"].corr(
                    table["effect_theirs"],
                    method="spearman",
                ),
            }
        )

    result = pd.DataFrame(records).sort_values("Cell type").reset_index(drop=True)
    result["Pearson r"] = result["Pearson r"].round(2)
    result["Spearman ρ"] = result["Spearman ρ"].round(2)
    return result


def calculate_cat_at_k(
    comparisons: dict[str, pd.DataFrame | None],
    *,
    method: str,
    include_region: bool,
) -> pd.DataFrame:
    """Calculate CAT@K from absolute effect-size rankings."""
    records = []

    for label, table in comparisons.items():
        if table is None:
            continue

        rank_primary = table["effect_ours"].abs().sort_values(ascending=False)
        rank_sun = table["effect_theirs"].abs().sort_values(ascending=False)

        parts = label.split("_", maxsplit=1)
        region = parts[0] if len(parts) == 2 else None
        cell_type = parts[1] if len(parts) == 2 else label

        for k in CAT_K_VALUES:
            if k > len(table):
                continue

            overlap = len(
                set(rank_primary.head(k).index)
                & set(rank_sun.head(k).index)
            ) / k

            record = {
                "celltype": cell_type,
                "K": k,
                "CAT@K": overlap,
                "method": method,
            }
            if include_region:
                record["region"] = region
            records.append(record)

    return pd.DataFrame(records)


def run_rank_based_pseudobulk(
    adata: ad.AnnData,
    *,
    cell_type_key: str,
    age_key: str,
    replicate_key: str,
    valid_cell_types: Iterable[str],
) -> dict[str, pd.DataFrame]:
    """Run the original replicate-mean Mann–Whitney workflow."""
    output = {}

    for cell_type in np.unique(adata.obs[cell_type_key]):
        if cell_type not in valid_cell_types:
            continue

        subset = adata[adata.obs[cell_type_key] == cell_type].copy()
        records = []

        for gene in subset.var_names:
            values = subset[:, gene].X
            expression = (
                values.toarray().ravel()
                if hasattr(values, "toarray")
                else np.asarray(values).ravel()
            )

            frame = pd.DataFrame(
                {
                    "exp": expression,
                    "age": subset.obs[age_key].values,
                    "replicate": subset.obs[replicate_key].values,
                }
            )
            pseudobulk = (
                frame.groupby(["replicate", "age"], as_index=False)["exp"].mean()
            )
            pseudobulk = pseudobulk.dropna(subset=["exp"])

            young = pseudobulk.loc[pseudobulk["age"] == "Yng", "exp"].values
            old = pseudobulk.loc[pseudobulk["age"] == "Old", "exp"].values

            if len(young) < 2 or len(old) < 2:
                continue

            _, p_value = mannwhitneyu(old, young, alternative="two-sided")
            effect = old.mean() - young.mean()

            records.append(
                {
                    "gene": gene,
                    "effect": effect,
                    "pval": p_value,
                }
            )

        if records:
            output[cell_type] = pd.DataFrame(records).set_index("gene")

    return output


def render_supplementary_table(
    table: pd.DataFrame,
    *,
    title: str,
    output_path: Path,
) -> None:
    """Render a correlation table as a publication-ready PDF."""
    figure, axis = plt.subplots(figsize=(14, 2))
    axis.axis("off")

    column_widths = [0.24, 0.10, 0.16, 0.17, 0.16, 0.17]
    if len(table.columns) != len(column_widths):
        column_widths = [1 / len(table.columns)] * len(table.columns)

    rendered = axis.table(
        cellText=table.values,
        colLabels=table.columns,
        loc="center",
        cellLoc="center",
        colWidths=column_widths,
    )
    rendered.auto_set_font_size(False)
    rendered.set_fontsize(9)
    rendered.scale(1.0, 1.4)

    for (row, _), cell in rendered.get_celld().items():
        cell.set_linewidth(0.4)
        if row == 0:
            cell.set_text_props(weight="bold")
            cell.set_height(cell.get_height() * 1.35)

    plt.title(title, pad=12)
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

## 3. Load and subset the Sun et al. dataset

In [ ]:
if ALDEX_RESULT_MODE == "recompute":
    sun = sc.read_h5ad(SUN_H5AD)
    
    required_columns = {
        "age",
        "celltype",
        "subregion",
        "region",
        "brain_section_label",
    }
    missing = required_columns - set(sun.obs.columns)
    if missing:
        raise KeyError(f"Sun AnnData is missing required columns: {sorted(missing)}")
    
    sun = sun[sun.obs["age"].isin(SUN_SELECTED_AGES)].copy()
    
    sun.obs["celltype"] = (
        sun.obs["celltype"]
        .astype(str)
        .str.replace(" ", "-", regex=False)
    )
    sun.obs["celltype"] = pd.Categorical(sun.obs["celltype"])
    
    sun.obs["numerical_age"] = sun.obs["age"].astype(float)
    sun.obs["age"] = np.where(
        sun.obs["numerical_age"] > 10,
        "Old",
        "Yng",
    )
    sun.obs["age"] = pd.Categorical(sun.obs["age"])
    
    print(sun)
    display(
        sun.obs.groupby(["age", "numerical_age"], observed=True)
        .size()
        .to_frame("n_cells")
    )
else:
    sun = None
    print("Published mode: skipping Sun AnnData loading and ALDEx input generation.")


## 4. Export Sun et al. counts and metadata for ALDEx

Two sets of inputs are exported:

1. Region-specific inputs for the selected cell types.
2. An all-region input containing all retained cells and cell types.

This replaces the erroneous subregion-based export in the draft notebook.

In [ ]:
if ALDEX_RESULT_MODE == "recompute":
    REGION_INPUT_DIR = DE_RESULT_DIR / "cross_study"
    REGION_INPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    for region in sun.obs["region"].cat.categories:
        region_data = sun[sun.obs["region"] == region].copy()
        region_data = region_data[
            region_data.obs["celltype"].isin(SUN_VALID_CELL_TYPES)
        ].copy()
    
        safe_region = (
            str(region)
            .replace("/", "-")
            .replace("_", "-")
        )
    
        de.prep_for_aldex2(
            region_data,
            str(REGION_INPUT_DIR),
            f"cell_type_{safe_region}",
            obs_key="celltype",
            count_layer=False,
        )


In [ ]:
if ALDEX_RESULT_MODE == "recompute":
    de.prep_for_aldex2(
        sun,
        str(REGION_INPUT_DIR),
        "cell_type_all",
        obs_key="celltype",
        count_layer=False,
    )


## 5. Postprocess Sun et al. ALDEx results

The Sun et al. raw ALDEx workbooks are postprocessed in this notebook. The primary MERFISH workbook is already postprocessed by the primary-dataset analysis and is only loaded here.

Sun et al. results are processed twice:

- significant results only, for signed summaries;
- all results, for quantitative cross-dataset comparisons.

In [ ]:
if ALDEX_RESULT_MODE == "recompute":
    sun_aldex_raw_files = sorted(glob(SUN_ALDEX_RAW_GLOB))

    sun_aldex_significant = postprocess_aldex2(
        aldex_results=sun_aldex_raw_files,
        output_dir=DE_RESULT_DIR,
        filename_prefix=SUN_ALDEX_RAW_PREFIX,
        filename_suffix=SUN_ALDEX_RAW_SUFFIX,
        output_stem="cross_study1_inftss_results",
        experimental_column="age_binary",
        pivot_filename="cross_study1_inftss_pivot_signed.xlsx",
        significant=True,
        make_pivot=True,
        signed=True,
    )

    sun_aldex_all = postprocess_aldex2(
        aldex_results=sun_aldex_raw_files,
        output_dir=DE_RESULT_DIR,
        filename_prefix=SUN_ALDEX_RAW_PREFIX,
        filename_suffix=SUN_ALDEX_RAW_SUFFIX,
        output_stem="cross_study1_ALL_inftss_results",
        experimental_column="age_binary",
        pivot_filename=None,
        significant=False,
        make_pivot=False,
        signed=False,
    )
elif ALDEX_RESULT_MODE == "published":
    sun_aldex_all_raw = load_pooled_aldex_spec(config.additional_data_dir, "sun_binary_celltype_region")
    sun_aldex_significant_raw = {key: table.loc[table["age_binary:pval.adj"] <= 0.05].copy() for key, table in sun_aldex_all_raw.items()}
    print(f"Loaded {len(sun_aldex_all_raw)} pooled Sun binary-age strata.")
else:
    raise ValueError("ALDEX_RESULT_MODE must be published or recompute")


## 6. Load postprocessed results and build cross-dataset comparisons

In [ ]:
# Sun et al. output generated immediately above.
if ALDEX_RESULT_MODE == "published":
    sun_aldex_results = index_result_tables(
        load_pooled_aldex_spec(config.additional_data_dir, "sun_binary_celltype_region")
    )
    primary_aldex_results = index_result_tables(
        load_pooled_aldex_spec(config.additional_data_dir, "primary_anterior_celltype_region")
    )
else:
    sun_aldex_results = read_excel_sheets(SUN_ALDEX_ALL)
    primary_aldex_results = read_excel_sheets(PRIMARY_ALDEX_ALL)

aldex_comparisons, common_genes_by_primary_key = build_aldex_comparisons(
    primary_aldex_results,
    sun_aldex_results,
)

valid_aldex_comparisons = {
    key: table
    for key, table in aldex_comparisons.items()
    if table is not None
}

print(f"ALDEx comparisons retained: {len(valid_aldex_comparisons)}")
print(sorted(valid_aldex_comparisons))

## 7. ALDEx sign concordance — Figure 4

In [ ]:
aldex_isocortex_sign_matrix = plot_sign_concordance(
    aldex_comparisons,
    title="ALDEx sign concordance: primary MERFISH vs Sun et al.",
    output_path=FIGURE_DIR / "fig4_sun_aldex_sign_concordance.pdf",
    key_filter="Isocortex",
)
aldex_isocortex_sign_matrix.to_csv(
    DE_RESULT_DIR / "sun_aldex_isocortex_sign_concordance.csv"
)

## 8. ALDEx effect-size correlation table

In [ ]:
aldex_correlation_table = correlation_table(aldex_comparisons)
display(aldex_correlation_table)

aldex_correlation_table.to_excel(
    DE_RESULT_DIR / "Sun_binary_ALDEx_effect_correlations.xlsx",
    index=False,
)

## 9. ALDEx CAT@K — Figure 4

In [ ]:
cat_df_aldex = calculate_cat_at_k(
    aldex_comparisons,
    method="ALDEx",
    include_region=True,
)

cat_df_aldex.to_csv(
    DE_RESULT_DIR / "sun_binary_aldex_cat_at_k.csv",
    index=False,
)

plt.figure(figsize=(6, 4))
for region in cat_df_aldex["region"].unique():
    subset = cat_df_aldex[cat_df_aldex["region"] == region]
    plt.plot(
        subset.groupby("K", as_index=False)["CAT@K"].mean()["K"],
        subset.groupby("K", as_index=False)["CAT@K"].mean()["CAT@K"],
        marker="o",
        label=region,
    )
plt.xlabel("K")
plt.ylabel("CAT@K")
plt.title("CAT@K across regions — ALDEx")
plt.grid(True)
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "fig4_sun_aldex_cat_at_k_by_region.pdf",
    dpi=600,
    bbox_inches="tight",
)
plt.show()

## 10. Adapted rank-based pseudobulk analysis

Both datasets begin from raw counts, are normalized to a target sum of 250, and are log-transformed before replicate-level mean expression is calculated. This matches the intended original analysis.

The Sun et al. comparison is restricted to CTX. The primary MERFISH comparison is restricted to anterior isocortex.

In [ ]:
if ALDEX_RESULT_MODE == "published":
    primary_rank_results = index_result_tables(
        load_pooled_strata(
            config.additional_data_dir / "Additional_File_4.xlsx",
            sheet_name="Anterior_isocortex_pseudobulk",
            stratum_column="sheet_name",
            gene_column="gene",
        )
    )
    primary_rank_results = {
        label.removeprefix("Isocortex_"): table
        for label, table in primary_rank_results.items()
    }
    sun_rank_results = index_result_tables(
        load_pooled_strata(
            config.additional_data_dir / "Additional_File_5.xlsx",
            sheet_name="Binary_age_isocortex_pseudobulk",
            stratum_column="Isocortex Celltype",
            gene_column="gene",
        )
    )
    sun_rank_results = {
        label.removeprefix("Isocortex_"): table
        for label, table in sun_rank_results.items()
    }
    print(
        f"Loaded deposited pseudobulk results: {len(primary_rank_results)} primary "
        f"and {len(sun_rank_results)} Sun cell types."
    )
elif ALDEX_RESULT_MODE == "recompute":
    # Sun et al. CTX subset
    sun_rank = sun[sun.obs["region"] == "CTX"].copy()
    sun_rank.obs["age_group"] = pd.Categorical(
        np.where(sun_rank.obs["numerical_age"] > 10, "Old", "Yng")
    )
    
    # Restore raw counts if a preserved layer exists; otherwise use current X,
    # matching the original downloaded object.
    if "raw_counts" in sun_rank.layers:
        sun_rank.X = sun_rank.layers["raw_counts"].copy()
    
    sc.pp.normalize_total(sun_rank, target_sum=250)
    sc.pp.log1p(sun_rank)
    
    sun_rank_results = run_rank_based_pseudobulk(
        sun_rank,
        cell_type_key="celltype",
        age_key="age_group",
        replicate_key="brain_section_label",
        valid_cell_types=SUN_VALID_CELL_TYPES,
    )
    
    
    # Primary anterior isocortex subset
    primary = sc.read_h5ad(PRIMARY_H5AD)
    primary.X = primary.layers["raw_counts"].copy()
    primary = primary[primary.obs["batch"].isin([1, 2, 3])].copy()
    primary = primary[
        primary.obs["ct_region"].astype(str).str.contains(
            "Isocortex",
            na=False,
        )
    ].copy()
    
    sc.pp.normalize_total(primary, target_sum=250)
    sc.pp.log1p(primary)
    
    primary_rank_results = run_rank_based_pseudobulk(
        primary,
        cell_type_key="cell_type",
        age_key="Age",
        replicate_key="sample",
        valid_cell_types=PRIMARY_TO_SUN_CELL_TYPE.keys(),
    )
else:
    raise ValueError("ALDEX_RESULT_MODE must be 'published' or 'recompute'.")


## 11. Save rank-based results

In [ ]:
write_excel_sheets(
    primary_rank_results,
    DE_RESULT_DIR / "maindata_anterior_isocortex_pseudobulk.xlsx",
)
write_excel_sheets(
    sun_rank_results,
    DE_RESULT_DIR / "sun_et_al_binary_isocortex_pseudobulk.xlsx",
)

## 12. Build rank-based cross-dataset comparisons

In [ ]:
rank_comparisons = {}

for primary_cell_type, sun_cell_type in PRIMARY_TO_SUN_CELL_TYPE.items():
    if primary_cell_type not in primary_rank_results:
        continue
    if sun_cell_type not in sun_rank_results:
        continue

    common_key = f"Isocortex_{primary_cell_type.lower()}"
    common_genes = common_genes_by_primary_key.get(common_key)

    if common_genes is None:
        raise KeyError(
            f"No ALDEx common-gene universe found for {common_key}"
        )

    primary_table = primary_rank_results[primary_cell_type]
    sun_table = sun_rank_results[sun_cell_type]

    shared = [
        gene
        for gene in common_genes
        if gene in primary_table.index and gene in sun_table.index
    ]

    comparison = pd.concat(
        [
            primary_table.loc[shared, "effect"],
            sun_table.loc[shared, "effect"],
        ],
        axis=1,
    )
    comparison.columns = ["effect_ours", "effect_theirs"]
    comparison = comparison.dropna()

    if comparison.shape[0] >= 2:
        rank_comparisons[f"Isocortex_{primary_cell_type}"] = comparison

print(sorted(rank_comparisons))

## 13. Rank-based sign concordance — Figure 4

In [ ]:
rank_sign_matrix = plot_sign_concordance(
    rank_comparisons,
    title="Rank-based sign concordance: primary MERFISH vs Sun et al.",
    output_path=FIGURE_DIR / "fig4_sun_rank_based_sign_concordance.pdf",
)
rank_sign_matrix.to_csv(
    DE_RESULT_DIR / "sun_rank_based_isocortex_sign_concordance.csv"
)

## 14. Rank-based effect-size correlation table

In [ ]:
rank_correlation_table = correlation_table(rank_comparisons)
display(rank_correlation_table)

rank_correlation_table.to_excel(
    DE_RESULT_DIR / "Sun_binary_rank_based_effect_correlations.xlsx",
    index=False,
)

## 15. Rank-based CAT@K

In [ ]:
cat_df_rank = calculate_cat_at_k(
    rank_comparisons,
    method="Rank-based",
    include_region=False,
)
cat_df_rank.to_csv(
    DE_RESULT_DIR / "sun_binary_rank_based_cat_at_k.csv",
    index=False,
)

## 16. Combined ALDEx and rank-based CAT@K — Figure 4

In [ ]:
aldex_isocortex_cat = (
    cat_df_aldex[cat_df_aldex["region"] == "Isocortex"]
    .drop(columns="region")
    .copy()
)

cat_df_all = pd.concat(
    [aldex_isocortex_cat, cat_df_rank],
    ignore_index=True,
)
cat_df_all["celltype"] = cat_df_all["celltype"].str.upper()

plt.figure(figsize=(12, 8))
sns.lineplot(
    data=cat_df_all,
    x="K",
    y="CAT@K",
    hue="celltype",
    style="method",
    markers=True,
    markersize=8,
    linewidth=2.5,
)
plt.xlabel("K", fontsize=14)
plt.ylabel("CAT@K", fontsize=14)
plt.title("CAT@K comparison: ALDEx vs rank-based method", fontsize=16)
plt.xticks(CAT_K_VALUES, fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True)
plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    frameon=False,
)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "fig4_sun_cat_at_k_aldex_vs_rank_based.pdf",
    dpi=600,
    bbox_inches="tight",
)
plt.show()

## 17. ΔCAT@K — Figure 4

In [ ]:
aldex_delta_source = aldex_isocortex_cat.assign(
    celltype=aldex_isocortex_cat["celltype"].str.upper()
)
rank_delta_source = cat_df_rank.assign(
    celltype=cat_df_rank["celltype"].str.upper()
)

delta_df = (
    aldex_delta_source
    .rename(columns={"CAT@K": "CAT_aldex"})
    .merge(
        rank_delta_source.rename(columns={"CAT@K": "CAT_rank_based"}),
        on=["celltype", "K"],
        how="inner",
    )
)

delta_df["delta_CAT"] = (
    delta_df["CAT_aldex"] - delta_df["CAT_rank_based"]
)
if delta_df.empty:
    raise ValueError("No matched ALDEx/rank-based CAT@K rows after label harmonization.")

delta_selected = delta_df[
    delta_df["K"].isin(DELTA_K_VALUES)
].copy()

plt.figure(figsize=(8, 4))
sns.lineplot(
    data=delta_selected,
    x="K",
    y="delta_CAT",
    hue="celltype",
    marker=None,
    linewidth=2.5,
)
plt.axhline(0, linestyle="--", color="black", linewidth=1)
plt.xticks(CAT_K_VALUES, fontsize=12)
plt.yticks(fontsize=12)
plt.xlabel("K", fontsize=14)
plt.ylabel("ΔCAT@K (ALDEx − rank-based)", fontsize=14)
plt.title("Relative CAT@K performance", fontsize=16)
plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    frameon=False,
    fontsize=12,
)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "fig4_sun_delta_cat_at_k.pdf",
    dpi=600,
    bbox_inches="tight",
)
plt.show()

delta_df.to_csv(
    DE_RESULT_DIR / "sun_binary_delta_cat_at_k.csv",
    index=False,
)

## 18. Supplemental effect-size concordance table

In [ ]:
combined_correlations = pd.concat(
    [
        aldex_correlation_table.assign(Method="ALDEx"),
        rank_correlation_table.assign(Method="Rank-based"),
    ],
    ignore_index=True,
)

combined_correlations = combined_correlations[
    ["Method", "Cell type", "N genes", "Pearson r", "Spearman ρ"]
]

combined_correlations.to_excel(
    DE_RESULT_DIR / "Sun_all_effect_correlations.xlsx",
    index=False,
)

render_supplementary_table(
    combined_correlations,
    title=(
        "Supplementary Table S3. "
        "Sun et al. dataset effect-size concordance metrics"
    ),
    output_path=TABLE_DIR / "Supplementary_Table_S3.pdf",
)

display(combined_correlations)